In [1]:
%%capture
!pip install -U lightautoml

In [2]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [3]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.tree import DecisionTreeRegressor

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [4]:
import torch
from lightautoml.automl.presets.tabular_presets import TabularAutoML, TabularUtilizedAutoML
from lightautoml.tasks import Task

In [5]:
N_THREADS = 4
N_FOLDS = 8
RANDOM_STATE = 42
TEST_SIZE = 0.2
TIMEOUT = 3600*100

numpy.random.seed(RANDOM_STATE)
torch.set_num_threads(N_THREADS)

train = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/train.csv')
test = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv').drop(columns=['id'])

In [6]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

In [7]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]] = train[i[0]].fillna('Unknown')
        train[i[0]]=normalize.fit_transform(encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1)).reshape(-1,1))
    else:
        train[i[0]].fillna(train[i[0]].mean(), inplace=True)
        train[i[0]]=normalize.fit_transform(numpy.array(train[i[0]]).reshape(-1,1))
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]] = test[i[0]].fillna('Unknown')
        test[i[0]]=normalize.fit_transform(numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1))).reshape(-1,1))
    else:
        test[i[0]].fillna(test[i[0]].mean(), inplace=True)
        test[i[0]]=normalize.fit_transform(numpy.array(test[i[0]]).reshape(-1,1))

In [8]:
train = train.drop(columns=['id'])

train_x = train.drop(columns=['efficiency'])
train_y = train['efficiency']

clf = DecisionTreeRegressor().fit(train_x, train_y)
model = SelectFromModel(clf, prefit=True)

weight = model.transform(train_x)
train['weight']=numpy.mean(weight, axis=1)
train['weight']

0        203.039323
1         86.820296
2        229.977199
3        251.490097
4         14.505244
            ...    
19995    172.135591
19996    107.826792
19997    231.231672
19998    217.507699
19999    271.596760
Name: weight, Length: 20000, dtype: float64

In [9]:
task = Task('reg')

In [10]:
model = TabularAutoML(
    task = task,
    timeout = TIMEOUT,
    cpu_limit = N_THREADS,
    general_params = {'use_algos':[['cb', 'linear_l2', 'gbm', 'lgb', 'lgb_tuned']]},
    selection_params={'mode' : 0},
    tuning_params = {'max_tuning_time': 3600},
    reader_params = {'n_jobs': N_THREADS, 'cv': N_FOLDS, 'random_state': RANDOM_STATE}
)

out_of_fold_predictions = model.fit_predict(
    train,
    roles = {
        'target': 'efficiency',
        #'drop': ['unique_id']
        'weights': 'weight'
    }, 
    verbose = 2
)

[07:25:52] Stdout logging level is INFO2.
[07:25:52] Task: reg

[07:25:52] Start automl preset with listed constraints:
[07:25:52] - time: 360000.00 seconds
[07:25:52] - CPU: 4 cores
[07:25:52] - memory: 16 GB

[07:25:52] Train data shape: (20000, 17)

[07:26:01] Layer 1 train process start. Time left 359990.63 secs
[07:26:01] Start fitting Lvl_0_Pipe_0_Mod_0_LinearL2 ...
[07:26:01] ===== Start working with fold 0 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[07:26:01] ===== Start working with fold 1 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[07:26:01] ===== Start working with fold 2 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[07:26:02] ===== Start working with fold 3 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[07:26:02] ===== Start working with fold 4 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[07:26:02] ===== Start working with fold 5 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[07:26:02] ===== Start working with fold 6 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[07:26:02] ===== Start working with fold 7 for Lvl_0_Pipe_

Optimization Progress: 100%|██████████| 101/101 [01:39<00:00,  1.02it/s, best_trial=83, best_value=2.69e+71]

[07:27:44] Hyperparameters optimization for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM completed
[07:27:44] The set of hyperparameters {'feature_fraction': 0.7259804360312447, 'num_leaves': 88, 'bagging_fraction': 0.602787847363802, 'min_sum_hessian_in_leaf': 0.007489408163616094, 'reg_alpha': 8.092824577516295e-06, 'reg_lambda': 0.00011699710737168014}
 achieve 268627981402301087755377778032536535515576153080768675704964433454301184.0000 mse
[07:27:44] Start fitting Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM ...
[07:27:44] ===== Start working with fold 0 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====


[07:27:44] ===== Start working with fold 1 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====
[07:27:45] ===== Start working with fold 2 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====
[07:27:45] ===== Start working with fold 3 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====
[07:27:45] ===== Start working with fold 4 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====
[07:27:46] ===== Start working with fold 5 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====
[07:27:47] ===== Start working with fold 6 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====
[07:27:47] ===== Start working with fold 7 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====
[07:27:47] Model Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM failed during ml_algo.fit_predict call.

Input contains infinity or a value too large for dtype('float32').
[07:27:47] Start fitting Lvl_0_Pipe_1_Mod_2_CatBoost ...
[07:27:47] ===== Start working with fold 0 for Lvl_0_Pipe_1_Mod_2_CatBoost =====
[07:27:47] Model Lvl_0_Pipe_1_Mod_2_CatBoost failed during ml_algo.fit_predict call.

catboost

In [11]:
#print(f'Concordance index (lifelines): {100*(1-numpy.sqrt(mean_squared_error(y_test, model.predict(X_test))))}')

In [12]:
id = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv')['id']
test_predictions = model.predict(test)
test_predictions

array([[-1.8454986e+35],
       [ 1.8249173e+23],
       [ 1.6878052e+23],
       ...,
       [ 1.6680472e+23],
       [ 1.6680229e+23],
       [ 1.6760378e+23]], dtype=float32)

In [13]:
submission = pandas.DataFrame({
    'id': id.values,
    'efficiency': test_predictions.data.reshape(-1),
})
submission

,id,efficiency
0,0,-1.845499e+35
1,1,1.824917e+23
2,2,1.687805e+23
3,3,1.668047e+23
4,4,1.676038e+23
...,...,...
11995,11995,1.687805e+23
11996,11996,1.668047e+23
11997,11997,1.668047e+23
11998,11998,1.668023e+23


In [14]:
submission.to_csv('submission.csv', index = False)